In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import xgboost as xgb
from sklearn.metrics import f1_score


In [15]:
# Loading the treated data
train_dataset = pd.read_csv('data/train_dataset_treated.csv')

In [16]:
# Defining the function that will use the model to complete the missing values
# Using the model to complete the test dataset

def complete_dataset(model_name, model):

    test_dataset = pd.read_csv('data/test_dataset_treated.csv')

    test_pred = model.predict(test_dataset)
    test_dataset['Transition'] = test_pred
    test_dataset.head()

    # Dropping all columns but the Transition column
    test_dataset.drop(test_dataset.columns.difference(['Transition']), axis=1, inplace=True)

    # Creating a RowId column to store the index, starting from 1
    test_dataset['RowId'] = np.arange(1, test_dataset.shape[0] + 1)

    # Placing the RowId column in the first position
    cols = test_dataset.columns.tolist()
    cols = cols[-1:] + cols[:-1]
    test_dataset = test_dataset[cols]

    # Transforming the Transition column back to its original values
    replace_map = {'Transition': {0: 'CN-CN', 1: 'AD-AD', 2: 'CN-MCI', 3: 'MCI-AD', 4: 'MCI-MCI'}}
    test_dataset.replace(replace_map, inplace=True)
    test_dataset.head()

    # Saving the test dataset to a csv file
    test_dataset.to_csv('test_predictions_' + model_name + '.csv', index=False)

## Random Forest

In [17]:
# Running a Random Forest Classifier
#model_name = 'random_forest'
X = train_dataset.drop('Transition', axis=1)
y = train_dataset['Transition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)

rfc = RandomForestClassifier(n_estimators=100, random_state=123)
rfc.fit(X_train, y_train)
rfc_pred = rfc.predict(X_test)

In [18]:
# Printing f1 score
print(f1_score(y_test, rfc_pred, average='weighted'))

0.46490445886280435


In [19]:
# Printing the confusion matrix
print("Confusion matrix")
print(confusion_matrix(y_test, rfc_pred))

Confusion matrix
[[21  1  0  2  6]
 [ 2  9  0 11  2]
 [ 0  0  0  2  2]
 [ 3  5  0  6  6]
 [ 3  2  0  2  7]]


In [20]:
# Printing the classification report
print("Classification report")
print(classification_report(y_test, rfc_pred))

Classification report
              precision    recall  f1-score   support

           0       0.72      0.70      0.71        30
           1       0.53      0.38      0.44        24
           2       0.00      0.00      0.00         4
           3       0.26      0.30      0.28        20
           4       0.30      0.50      0.38        14

    accuracy                           0.47        92
   macro avg       0.36      0.38      0.36        92
weighted avg       0.48      0.47      0.46        92



c:\Users\Utilizador\miniconda3\envs\daa\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Utilizador\miniconda3\envs\daa\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Utilizador\miniconda3\envs\daa\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(resu

In [21]:
# Completing the test dataset
complete_dataset('random_forest', rfc)

## XGBoost

In [22]:
#model_name = 'xgboost'
X = train_dataset.drop('Transition', axis=1)
y = train_dataset['Transition']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=123)

xgbc = xgb.XGBClassifier(max_depth = 1, objective='req:squarederror', random_state=123, learning_rate=0.2, n_estimators=52)
xgbc.fit(X_train, y_train)
xgbc_pred = xgbc.predict(X_test)

# Printing f1 score
print(f1_score(y_test, xgbc_pred, average='weighted'))

0.46636434217955947


In [23]:
# Printing f1 score
print(f1_score(y_test, xgbc_pred, average='weighted'))

0.46636434217955947


In [24]:
# Printing the confusion matrix
print("Confusion matrix")
print(confusion_matrix(y_test, xgbc_pred))

Confusion matrix
[[22  3  0  1  4]
 [ 3 10  0  8  3]
 [ 3  1  0  0  0]
 [ 2  3  0  9  6]
 [ 4  3  0  4  3]]


In [25]:
# Printing the classification report
print("Classification report")
print(classification_report(y_test, xgbc_pred))

Classification report
              precision    recall  f1-score   support

           0       0.65      0.73      0.69        30
           1       0.50      0.42      0.45        24
           2       0.00      0.00      0.00         4
           3       0.41      0.45      0.43        20
           4       0.19      0.21      0.20        14

    accuracy                           0.48        92
   macro avg       0.35      0.36      0.35        92
weighted avg       0.46      0.48      0.47        92



c:\Users\Utilizador\miniconda3\envs\daa\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Utilizador\miniconda3\envs\daa\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Utilizador\miniconda3\envs\daa\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(resu

In [26]:
# Completing the test dataset
complete_dataset('xgboost', xgbc)